# Full PINN Prior-Sensitivity and Observation-Stress Test

This is the full-PINN version of the synthetic stress notebook. It replaces the fast inverse-regression diagnostic with a trainable physics-informed neural network for regional SIRD dynamics.

The model learns:

- neural state trajectories $S_p(t), I_p(t), R_p(t), D_p(t)$,
- global transmission scale $\alpha$,
- row-stochastic geographic coupling $C$,
- optionally reporting factors and clinical rates.

The default version is a **discrete-time PINN**: it enforces the SIRD residual by finite differences on the time grid. This is much faster and easier to run in a notebook than autograd-through-time collocation, while still training the neural state model and physical parameters jointly. It is suitable for the synthetic prior-sensitivity and observation-stress experiment you need for the paper.

## How to use this notebook

Run the notebook as-is first. Then change only the options in the next cell.

Recommended paper ablations:

1. `run_prior_grid = True`: shows how inferred geography changes under different priors.
2. `run_observation_grid = True`: shows how noisy, sparse, cases-only, and reporting-biased observations affect coupling recovery.
3. `learn_reporting = True`: optional stress test showing that reporting and geography are confounded unless reporting is externally constrained.

In [ ]:
from __future__ import annotations

import math
import time
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
import torch
from torch import nn

# Critical for fast CPU execution with small networks in notebooks.
torch.set_num_threads(1)

CONFIG = {
    # Output location.
    "output_dir": "artifacts/synthetic",

    # Synthetic setup.
    "seed": 2026,
    "P": 4,
    "T": 90,

    # Runtime.
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "state_net_width": 32,
    "state_net_depth": 2,
    "learning_rate": 3e-3,
    "epochs": 800,
    "warmup_epochs": 200,
    "print_every": 200,

    # Main single run.
    "selected_obs_scenario": "clean_full",
    "selected_prior_scenario": "distance_medium",

    # Optional full-PINN ablations. These run quickly with the default finite-difference PINN.
    "run_prior_grid": True,
    "run_observation_grid": True,
    "grid_epochs": 400,
    "grid_warmup_epochs": 100,
    "grid_seeds": [0],

    # Model options.
    "learn_rates": False,
    "learn_reporting": False,

    # Loss weights.
    "loss_weights": {
        "state_data": 1.0,
        "incidence_data": 20.0,
        "physics": 1.0,
        "initial_condition": 10.0,
        "reporting_prior": 0.10,
    },

    # Prior loss: lambda_C * mean(((C-C0)/prior_scale)^2)
    "prior_scale": 0.10,
}

OUTDIR = Path(CONFIG["output_dir"])
FIGDIR = OUTDIR / "figures"
OUTDIR.mkdir(exist_ok=True)
FIGDIR.mkdir(exist_ok=True)

np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

device = torch.device(CONFIG["device"])
dtype = torch.float32

print("Device:", device)
print("Outputs:", OUTDIR.resolve())

## 1. Synthetic regional SIRD truth

The true model is:

\[
\lambda_p(t)=\alpha\sum_q C_{pq}\frac{I_q(t)}{S_q(t)+I_q(t)+R_q(t)},
\]

\[
\dot S_p=-S_p\lambda_p,
\quad
\dot I_p=S_p\lambda_p-(\gamma_p+\mu_p)I_p,
\quad
\dot R_p=\gamma_p I_p,
\quad
\dot D_p=\mu_p I_p.
\]

The geographic object of interest is the row-stochastic source-composition matrix $C$.

In [ ]:
P = CONFIG["P"]
T = CONFIG["T"]
dt = 1.0
region_names = [f"R{p}" for p in range(P)]

coords_np = np.array([
    [0.0, 0.0],
    [1.0, 0.2],
    [0.4, 1.2],
    [1.4, 1.1],
], dtype=np.float32)

N_np = np.array([1.00, 0.78, 0.62, 0.55], dtype=np.float32)
I0_np = np.array([0.0040, 0.0015, 0.0008, 0.0004], dtype=np.float32)
R0_np = np.zeros(P, dtype=np.float32)
D0_np = np.zeros(P, dtype=np.float32)
S0_np = N_np - I0_np - R0_np - D0_np

alpha_true = 0.42
gamma_true_np = np.array([0.100, 0.095, 0.105, 0.098], dtype=np.float32)
mu_true_np = np.array([0.0030, 0.0035, 0.0027, 0.0032], dtype=np.float32)

C_true_np = np.array([
    [0.80, 0.12, 0.06, 0.02],
    [0.10, 0.78, 0.08, 0.04],
    [0.05, 0.12, 0.78, 0.05],
    [0.02, 0.07, 0.11, 0.80],
], dtype=np.float32)

assert np.allclose(C_true_np.sum(axis=1), 1.0)


def simulate_sird(alpha: float, C: np.ndarray, gamma: np.ndarray, mu: np.ndarray, T: int = T, dt: float = 1.0) -> dict[str, np.ndarray]:
    """RK4 simulator for the synthetic regional SIRD truth."""
    S = np.zeros((T, P), dtype=np.float32)
    I = np.zeros((T, P), dtype=np.float32)
    R = np.zeros((T, P), dtype=np.float32)
    D = np.zeros((T, P), dtype=np.float32)
    S[0], I[0], R[0], D[0] = S0_np, I0_np, R0_np, D0_np
    y = np.r_[S0_np, I0_np, R0_np, D0_np].astype(np.float32)

    def rhs(y_vec: np.ndarray) -> np.ndarray:
        S_ = y_vec[:P]
        I_ = y_vec[P:2*P]
        R_ = y_vec[2*P:3*P]
        live = np.maximum(S_ + I_ + R_, 1e-12)
        prevalence = I_ / live
        incidence = S_ * (alpha * (C @ prevalence))
        return np.r_[
            -incidence,
            incidence - (gamma + mu) * I_,
            gamma * I_,
            mu * I_,
        ].astype(np.float32)

    for t in range(1, T):
        k1 = rhs(y)
        k2 = rhs(y + 0.5 * dt * k1)
        k3 = rhs(y + 0.5 * dt * k2)
        k4 = rhs(y + dt * k3)
        y = np.maximum(y + dt * (k1 + 2*k2 + 2*k3 + k4) / 6.0, 0.0)
        S[t], I[t], R[t], D[t] = y[:P], y[P:2*P], y[2*P:3*P], y[3*P:4*P]

    prevalence = I / np.maximum(S + I + R, 1e-12)
    incidence = S * (alpha * (prevalence @ C.T))
    return {
        "S": S,
        "I": I,
        "R": R,
        "D": D,
        "incidence": incidence,
        "new_R": gamma[None, :] * I,
        "new_D": mu[None, :] * I,
    }

truth = simulate_sird(alpha_true, C_true_np, gamma_true_np, mu_true_np)

# Torch constants.
N_t = torch.tensor(N_np, device=device, dtype=dtype)
S0_t = torch.tensor(S0_np, device=device, dtype=dtype)
I0_t = torch.tensor(I0_np, device=device, dtype=dtype)
R0_t = torch.tensor(R0_np, device=device, dtype=dtype)
D0_t = torch.tensor(D0_np, device=device, dtype=dtype)
gamma_true_t = torch.tensor(gamma_true_np, device=device, dtype=dtype)
mu_true_t = torch.tensor(mu_true_np, device=device, dtype=dtype)

print("True alpha:", alpha_true)
print("True C:\n", C_true_np)
print("Peak I by region:", truth["I"].max(axis=0))

In [ ]:
# Plot truth.
t = np.arange(T)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharex=True)
for p, name in enumerate(region_names):
    axes[0].plot(t, truth["I"][:, p], label=name)
    axes[1].plot(t, truth["R"][:, p], label=name)
    axes[2].plot(t, truth["D"][:, p], label=name)
for ax, title in zip(axes, ["Infectious", "Recovered", "Deaths"]):
    ax.set_title(title)
    ax.set_xlabel("Day")
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel("Scaled count")
axes[0].legend(fontsize=8)
fig.tight_layout()
_img = FIGDIR / "truth_trajectories.png"
fig.savefig(_img, dpi=180, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(_img)))

## 2. Prior and observation scenarios

The prior enters the full PINN as:

\[
\lambda_C \; \mathrm{mean}\left(\frac{C-C_0}{s_C}\right)^2.
\]

The default `distance_medium` prior is close but not identical to the truth. The intentionally bad priors are included to test whether the inferred coupling matrix is robust or prior-driven.

In [ ]:
def pairwise_dist(x: np.ndarray) -> np.ndarray:
    d = x[:, None, :] - x[None, :, :]
    return np.sqrt((d * d).sum(axis=-1))


def make_prior(kind: str = "distance", diag_mass: float = 0.80, length_scale: float = 1.0,
               lambda_C: float = 0.20, alpha_init: float = 0.35) -> dict:
    dist = pairwise_dist(coords_np)
    if kind == "distance":
        W = np.exp(-dist / max(length_scale, 1e-8))
    elif kind == "uniform":
        W = np.ones((P, P), dtype=np.float32)
    elif kind == "anti_distance":
        W = np.exp(dist / max(length_scale, 1e-8))
    else:
        raise ValueError(f"Unknown prior kind: {kind}")
    np.fill_diagonal(W, 0.0)

    C0 = np.zeros((P, P), dtype=np.float32)
    for p in range(P):
        C0[p, p] = diag_mass
        C0[p] += (1.0 - diag_mass) * W[p] / max(W[p].sum(), 1e-12)
    return {"C0": C0.astype(np.float32), "lambda_C": lambda_C, "alpha_init": alpha_init, "kind": kind}


PRIOR_SCENARIOS = {
    "no_prior": make_prior("distance", diag_mass=0.80, length_scale=1.0, lambda_C=0.00),
    "distance_weak": make_prior("distance", diag_mass=0.80, length_scale=1.0, lambda_C=0.03),
    "distance_medium": make_prior("distance", diag_mass=0.80, length_scale=1.0, lambda_C=0.20),
    "too_diagonal": make_prior("distance", diag_mass=0.93, length_scale=1.0, lambda_C=0.20),
    "too_mixed": make_prior("distance", diag_mass=0.60, length_scale=1.0, lambda_C=0.20),
    "wrong_strong": make_prior("anti_distance", diag_mass=0.55, length_scale=2.5, lambda_C=1.00),
}


def add_noise(x: np.ndarray, rel: float = 0.01, seed: int = 0) -> np.ndarray:
    rg = np.random.default_rng(seed)
    sd = 1e-6 + rel * np.maximum(np.abs(x), 1e-8)
    return np.clip(x + rg.normal(0.0, sd, size=x.shape), 0.0, None).astype(np.float32)


def make_observations(name: str, seed: int = 0) -> dict[str, np.ndarray | None]:
    """Create observations for one stress scenario.

    Missing values are stored as NaN and ignored in the data loss.
    """
    if name == "clean_full":
        return {k: add_noise(truth[k], rel=0.005, seed=seed+i) for i, k in enumerate(["I", "R", "D", "incidence"])}

    if name == "noisy_full":
        return {k: add_noise(truth[k], rel=0.030, seed=seed+i) for i, k in enumerate(["I", "R", "D", "incidence"])}

    if name == "weekly_sparse":
        obs = {k: add_noise(truth[k], rel=0.020, seed=seed+i) for i, k in enumerate(["I", "R", "D", "incidence"])}
        mask = np.zeros((T, P), dtype=bool)
        mask[::7, :] = True
        mask[0, :] = True
        for k in list(obs.keys()):
            arr = obs[k].copy()
            arr[~mask] = np.nan
            obs[k] = arr
        return obs

    if name == "reporting_bias":
        q = np.array([1.00, 0.70, 0.55, 0.90], dtype=np.float32)
        return {
            "incidence": add_noise(truth["incidence"] * q[None, :], rel=0.025, seed=seed),
            "I": None,
            "R": None,
            "D": None,
            "reporting_true": q,
        }

    if name == "cases_only_proxy":
        return {
            "incidence": add_noise(truth["incidence"], rel=0.025, seed=seed),
            "I": None,
            "R": None,
            "D": None,
        }

    raise ValueError(f"Unknown observation scenario: {name}")


OBS_SCENARIOS = ["clean_full", "noisy_full", "weekly_sparse", "reporting_bias", "cases_only_proxy"]

for name, p in PRIOR_SCENARIOS.items():
    print(f"{name:16s} lambda={p['lambda_C']:.2f}, mean diag={np.diag(p['C0']).mean():.2f}")
print("Observation scenarios:", OBS_SCENARIOS)

In [ ]:
def heatmap(ax, M, title, vmin=0.0, vmax=1.0, fontsize=8):
    im = ax.imshow(M, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.set_xticks(range(P))
    ax.set_yticks(range(P))
    ax.set_xlabel("source q")
    ax.set_ylabel("recipient p")
    ax.set_aspect("equal")
    for i in range(P):
        for j in range(P):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=fontsize)
    return im

# Prior overview with dedicated colorbar column.
items = [
    ("true C", C_true_np),
    ("distance_medium", PRIOR_SCENARIOS["distance_medium"]["C0"]),
    ("too_diagonal", PRIOR_SCENARIOS["too_diagonal"]["C0"]),
    ("wrong_strong", PRIOR_SCENARIOS["wrong_strong"]["C0"]),
]
fig = plt.figure(figsize=(13.2, 3.1))
gs = fig.add_gridspec(1, 5, width_ratios=[1, 1, 1, 1, 0.08], wspace=0.35)
axes = [fig.add_subplot(gs[0, j]) for j in range(4)]
cax = fig.add_subplot(gs[0, 4])
for ax, (title, M) in zip(axes, items):
    im = heatmap(ax, M, title)
cbar = fig.colorbar(im, cax=cax)
cbar.set_label("coupling weight", rotation=270, labelpad=12)
_img = FIGDIR / "prior_overview.png"
fig.savefig(_img, dpi=180, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(_img)))

## 3. Full PINN implementation

The neural network maps normalized time $\tau\in[0,1]$ to regional compartment fractions. A softmax over the four compartments enforces:

\[
S_p(t)+I_p(t)+R_p(t)+D_p(t)=N_p,
\quad S,I,R,D\ge 0.
\]

Coupling is parameterized by row-wise softmax logits, so $C_{pq}\ge0$ and $\sum_q C_{pq}=1$.

Training is staged:

1. Warmup: freeze $C$ and fit state trajectories, $\alpha$, and physics residual.
2. Unfreeze: learn $C$ under the prior and observation losses.

In [ ]:
def to_tensors(obs: dict[str, np.ndarray | None]):
    obs_tensors = {}
    masks = {}
    scales = {}
    for k, v in obs.items():
        if k == "reporting_true" or v is None:
            continue
        finite_mask = np.isfinite(v)
        obs_tensors[k] = torch.tensor(np.nan_to_num(v, nan=0.0), device=device, dtype=dtype)
        masks[k] = torch.tensor(finite_mask, device=device, dtype=torch.bool)
        finite = v[finite_mask]
        scales[k] = max(float(np.nanmax(np.abs(finite))) if finite.size else 1e-5, 1e-5)
    return obs_tensors, masks, scales


class StateNet(nn.Module):
    def __init__(self, width: int = 32, depth: int = 2):
        super().__init__()
        layers = []
        d = 1
        for _ in range(depth):
            layers += [nn.Linear(d, width), nn.Tanh()]
            d = width
        layers += [nn.Linear(d, P * 4)]
        self.net = nn.Sequential(*layers)

    def forward(self, tau: torch.Tensor, N: torch.Tensor):
        logits = self.net(tau).reshape(-1, P, 4)
        frac = torch.softmax(logits, dim=-1)
        states = frac * N[None, :, None]
        return states[..., 0], states[..., 1], states[..., 2], states[..., 3]


class RegionalSIRDPINN(nn.Module):
    def __init__(self, prior: dict, learn_rates: bool = False, learn_reporting: bool = False):
        super().__init__()
        self.learn_rates = learn_rates
        self.learn_reporting = learn_reporting
        self.state_net = StateNet(CONFIG["state_net_width"], CONFIG["state_net_depth"])

        alpha_lo, alpha_hi = 0.05, 1.20
        z = (prior["alpha_init"] - alpha_lo) / (alpha_hi - alpha_lo)
        self.raw_alpha = nn.Parameter(torch.tensor(math.log(z / (1 - z)), device=device, dtype=dtype))

        C0 = torch.tensor(prior["C0"], device=device, dtype=dtype)
        self.coupling_logits = nn.Parameter(torch.log(C0 + 1e-8))

        if learn_rates:
            self.raw_gamma = nn.Parameter(torch.zeros(P, device=device, dtype=dtype))
            self.raw_mu = nn.Parameter(torch.zeros(P, device=device, dtype=dtype))
        if learn_reporting:
            self.raw_reporting = nn.Parameter(torch.zeros(P, device=device, dtype=dtype))

    def alpha(self):
        return 0.05 + (1.20 - 0.05) * torch.sigmoid(self.raw_alpha)

    def C(self):
        return torch.softmax(self.coupling_logits, dim=1)

    def gamma(self):
        if self.learn_rates:
            return (1/28) + (1/3 - 1/28) * torch.sigmoid(self.raw_gamma)
        return gamma_true_t

    def mu(self):
        if self.learn_rates:
            return 1e-5 + (0.05 - 1e-5) * torch.sigmoid(self.raw_mu)
        return mu_true_t

    def reporting(self):
        if self.learn_reporting:
            return 0.30 + (1.20 - 0.30) * torch.sigmoid(self.raw_reporting)
        return torch.ones(P, device=device, dtype=dtype)

    def states(self, tau: torch.Tensor):
        return self.state_net(tau, N_t)

    def incidence_from_states(self, S: torch.Tensor, I: torch.Tensor, R: torch.Tensor):
        prevalence = I / torch.clamp(S + I + R, min=1e-10)
        return S * (self.alpha() * (prevalence @ self.C().T))


def train_pinn(obs_name: str, prior_name: str, seed: int = 0, epochs: int | None = None,
               warmup_epochs: int | None = None, verbose: bool = True):
    if epochs is None:
        epochs = CONFIG["epochs"]
    if warmup_epochs is None:
        warmup_epochs = CONFIG["warmup_epochs"]

    torch.manual_seed(CONFIG["seed"] + seed)
    obs = make_observations(obs_name, seed=1000 + seed)
    obs_tensors, masks, scales = to_tensors(obs)
    prior = PRIOR_SCENARIOS[prior_name]

    model = RegionalSIRDPINN(
        prior,
        learn_rates=CONFIG["learn_rates"],
        learn_reporting=CONFIG["learn_reporting"],
    ).to(device)

    C0_t = torch.tensor(prior["C0"], device=device, dtype=dtype)
    lambda_C = prior["lambda_C"]
    t_grid = torch.linspace(0, 1, T, device=device, dtype=dtype).reshape(-1, 1)

    phys_scale_np = np.maximum(
        np.stack([
            truth["incidence"].max(axis=0),
            truth["incidence"].max(axis=0),
            truth["new_R"].max(axis=0),
            truth["new_D"].max(axis=0),
        ]),
        1e-5,
    ).astype(np.float32)
    phys_scale = torch.tensor(phys_scale_np, device=device, dtype=dtype)
    weights = CONFIG["loss_weights"]

    def loss_fn():
        S, I, R, D = model.states(t_grid)
        incidence = model.incidence_from_states(S, I, R)

        data_loss = torch.tensor(0.0, device=device, dtype=dtype)
        for k, pred, w in [
            ("I", I, weights["state_data"]),
            ("R", R, weights["state_data"]),
            ("D", D, weights["state_data"]),
            ("incidence", incidence * model.reporting()[None, :], weights["incidence_data"]),
        ]:
            if k in obs_tensors:
                m = masks[k]
                data_loss = data_loss + w * torch.mean(((pred[m] - obs_tensors[k][m]) / scales[k]) ** 2)

        ic_loss = (
            torch.mean(((S[0] - S0_t) / N_t) ** 2)
            + torch.mean(((I[0] - I0_t) / (I0_t.max() + 1e-4)) ** 2)
            + torch.mean(((R[0] - R0_t) / N_t) ** 2)
            + torch.mean(((D[0] - D0_t) / N_t) ** 2)
        )

        dS = (S[1:] - S[:-1]) / dt
        dI = (I[1:] - I[:-1]) / dt
        dR = (R[1:] - R[:-1]) / dt
        dD = (D[1:] - D[:-1]) / dt
        I_left = I[:-1]
        incidence_left = incidence[:-1]
        gamma = model.gamma()
        mu = model.mu()
        residual = torch.stack([
            dS + incidence_left,
            dI - incidence_left + (gamma + mu)[None, :] * I_left,
            dR - gamma[None, :] * I_left,
            dD - mu[None, :] * I_left,
        ], dim=0)
        physics_loss = torch.mean((residual / phys_scale[:, None, :]) ** 2)

        prior_loss = lambda_C * torch.mean(((model.C() - C0_t) / CONFIG["prior_scale"]) ** 2)
        reporting_prior = torch.tensor(0.0, device=device, dtype=dtype)
        if model.learn_reporting:
            reporting_prior = weights["reporting_prior"] * torch.mean(((model.reporting() - 1.0) / 0.15) ** 2)

        total = (
            data_loss
            + weights["initial_condition"] * ic_loss
            + weights["physics"] * physics_loss
            + prior_loss
            + reporting_prior
        )
        return total, {
            "data": data_loss,
            "ic": ic_loss,
            "phys": physics_loss,
            "prior": prior_loss,
            "reporting_prior": reporting_prior,
        }

    # Stage 1: freeze geographic coupling.
    model.coupling_logits.requires_grad_(False)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=CONFIG["learning_rate"])

    history = []
    start = time.time()
    for ep in range(epochs + 1):
        if ep == warmup_epochs:
            model.coupling_logits.requires_grad_(True)
            opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=CONFIG["learning_rate"] * 0.5)

        opt.zero_grad(set_to_none=True)
        total, losses = loss_fn()
        total.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()

        if ep % CONFIG["print_every"] == 0 or ep == epochs:
            row = {
                "epoch": ep,
                "total": float(total.detach().cpu()),
                **{k: float(v.detach().cpu()) for k, v in losses.items()},
                "alpha": float(model.alpha().detach().cpu()),
            }
            history.append(row)
            if verbose:
                print(
                    f"{obs_name}/{prior_name} seed={seed} epoch={ep:4d} "
                    f"total={row['total']:.4g} data={row['data']:.3g} "
                    f"phys={row['phys']:.3g} alpha={row['alpha']:.3f}"
                )

    with torch.no_grad():
        S, I, R, D = model.states(t_grid)
        incidence = model.incidence_from_states(S, I, R)
        C_hat = model.C().detach().cpu().numpy()
        alpha_hat = float(model.alpha().detach().cpu())
        reporting_hat = model.reporting().detach().cpu().numpy()
        states = {
            "S": S.detach().cpu().numpy(),
            "I": I.detach().cpu().numpy(),
            "R": R.detach().cpu().numpy(),
            "D": D.detach().cpu().numpy(),
            "incidence": incidence.detach().cpu().numpy(),
        }

    metrics = {
        "obs_scenario": obs_name,
        "prior_scenario": prior_name,
        "seed": seed,
        "seconds": time.time() - start,
        "alpha_hat": alpha_hat,
        "alpha_abs_error": abs(alpha_hat - alpha_true),
        "C_fro_error": float(np.linalg.norm(C_hat - C_true_np)),
        "C_max_abs_error": float(np.max(np.abs(C_hat - C_true_np))),
        "C_mean_abs_error": float(np.mean(np.abs(C_hat - C_true_np))),
        "diag_mean_hat": float(np.mean(np.diag(C_hat))),
        "offdiag_mass_hat": float(np.mean(1.0 - np.diag(C_hat))),
        "reporting_hat_mean": float(reporting_hat.mean()),
        "final_total_loss": history[-1]["total"],
        "final_data_loss": history[-1]["data"],
        "final_phys_loss": history[-1]["phys"],
    }

    return {
        "model": model,
        "history": pd.DataFrame(history),
        "metrics": metrics,
        "C_hat": C_hat,
        "alpha_hat": alpha_hat,
        "reporting_hat": reporting_hat,
        "states": states,
        "obs": obs,
        "prior": prior,
    }

## 4. Run one full-PINN fit

This run is the easiest one to debug. Start here before running larger grids.

In [ ]:
selected_result = train_pinn(
    CONFIG["selected_obs_scenario"],
    CONFIG["selected_prior_scenario"],
    seed=0,
    epochs=CONFIG["epochs"],
    warmup_epochs=CONFIG["warmup_epochs"],
    verbose=True,
)

selected_metrics = pd.DataFrame([selected_result["metrics"]])
selected_metrics.to_csv(OUTDIR / "selected_run_metrics.csv", index=False)
selected_metrics

In [ ]:
# Plot loss history.
hist = selected_result["history"]
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.semilogy(hist["epoch"], hist["total"], marker="o", label="total")
ax.semilogy(hist["epoch"], hist["data"], marker="o", label="data")
ax.semilogy(hist["epoch"], hist["phys"], marker="o", label="physics")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Selected full-PINN training history")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
_img = FIGDIR / "selected_training_history.png"
fig.savefig(_img, dpi=180, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(_img)))

In [ ]:
# Coupling heatmaps for the selected run, using a dedicated colorbar column.
items = [
    ("true C", C_true_np),
    ("prior C0", selected_result["prior"]["C0"]),
    ("PINN estimate", selected_result["C_hat"]),
]
fig = plt.figure(figsize=(10.2, 3.1))
gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 0.08], wspace=0.35)
axes = [fig.add_subplot(gs[0, j]) for j in range(3)]
cax = fig.add_subplot(gs[0, 3])
for ax, (title, M) in zip(axes, items):
    im = heatmap(ax, M, title)
cbar = fig.colorbar(im, cax=cax)
cbar.set_label("coupling weight", rotation=270, labelpad=12)
_img = FIGDIR / "selected_coupling_heatmaps.png"
fig.savefig(_img, dpi=180, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(_img)))

In [ ]:
# Trajectory overlay for selected run.
states = selected_result["states"]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharex=True)
for p, name in enumerate(region_names):
    axes[0].plot(t, truth["I"][:, p], linestyle="-", label=f"{name} truth")
    axes[0].plot(t, states["I"][:, p], linestyle="--")
    axes[1].plot(t, truth["R"][:, p], linestyle="-")
    axes[1].plot(t, states["R"][:, p], linestyle="--")
    axes[2].plot(t, truth["D"][:, p], linestyle="-")
    axes[2].plot(t, states["D"][:, p], linestyle="--")
for ax, title in zip(axes, ["I", "R", "D"]):
    ax.set_title(f"{title}: truth solid, PINN dashed")
    ax.set_xlabel("Day")
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel("Scaled count")
axes[0].legend(fontsize=7, ncols=2)
fig.tight_layout()
_img = FIGDIR / "selected_trajectory_overlay.png"
fig.savefig(_img, dpi=180, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(_img)))

## 5. Prior-sensitivity grid with the full PINN

This grid holds the observation scenario fixed and changes the prior. If the recovered $C$ changes substantially across plausible priors while data/physics losses remain small, the geographic coupling is **prior-sensitive** and should not be over-interpreted.

In [ ]:
prior_grid_df = pd.DataFrame()
prior_grid_results = {}

if CONFIG["run_prior_grid"]:
    rows = []
    obs_name = CONFIG["selected_obs_scenario"]
    for prior_name, seed in itertools.product(PRIOR_SCENARIOS.keys(), CONFIG["grid_seeds"]):
        result = train_pinn(
            obs_name,
            prior_name,
            seed=seed,
            epochs=CONFIG["grid_epochs"],
            warmup_epochs=CONFIG["grid_warmup_epochs"],
            verbose=False,
        )
        rows.append(result["metrics"])
        prior_grid_results[(prior_name, seed)] = result
        print(f"prior={prior_name:16s} C_fro={result['metrics']['C_fro_error']:.3f} alpha={result['metrics']['alpha_hat']:.3f}")

    prior_grid_df = pd.DataFrame(rows).sort_values("C_fro_error")
    prior_grid_df.to_csv(OUTDIR / "full_pinn_prior_sensitivity.csv", index=False)

prior_grid_df

In [ ]:
if len(prior_grid_df):
    fig, ax = plt.subplots(figsize=(8.2, 3.6))
    plot_df = prior_grid_df.sort_values("C_fro_error")
    ax.bar(plot_df["prior_scenario"], plot_df["C_fro_error"])
    ax.set_ylabel(r"Coupling error $\|\hat C-C^*\|_F$")
    ax.set_title(f"Full-PINN prior sensitivity, obs={CONFIG['selected_obs_scenario']}")
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    _img = FIGDIR / "full_pinn_prior_sensitivity.png"
    fig.savefig(_img, dpi=180, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(_img)))

## 6. Observation-stress grid with the full PINN

This grid holds the prior fixed and changes the observations. The key warning case is `reporting_bias`: it fits biased incidence without external reporting calibration. If this produces a plausible loss but distorted $C$, that supports the paper's identifiability argument.

In [ ]:
obs_grid_df = pd.DataFrame()
obs_grid_results = {}

if CONFIG["run_observation_grid"]:
    rows = []
    prior_name = CONFIG["selected_prior_scenario"]
    for obs_name, seed in itertools.product(OBS_SCENARIOS, CONFIG["grid_seeds"]):
        result = train_pinn(
            obs_name,
            prior_name,
            seed=seed,
            epochs=CONFIG["grid_epochs"],
            warmup_epochs=CONFIG["grid_warmup_epochs"],
            verbose=False,
        )
        rows.append(result["metrics"])
        obs_grid_results[(obs_name, seed)] = result
        print(f"obs={obs_name:24s} C_fro={result['metrics']['C_fro_error']:.3f} alpha={result['metrics']['alpha_hat']:.3f}")

    obs_grid_df = pd.DataFrame(rows).sort_values("C_fro_error")
    obs_grid_df.to_csv(OUTDIR / "full_pinn_observation_stress.csv", index=False)

obs_grid_df

In [ ]:
if len(obs_grid_df):
    plot_df = obs_grid_df.sort_values("C_fro_error")

    fig, axes = plt.subplots(1, 2, figsize=(12.2, 3.8))

    axes[0].bar(plot_df["obs_scenario"], plot_df["C_fro_error"])
    axes[0].set_ylabel(r"Coupling error $\|\hat C-C^*\|_F$")
    axes[0].set_title("Coupling recovery")
    axes[0].tick_params(axis="x", rotation=35)
    axes[0].grid(axis="y", alpha=0.25)

    loss_df = obs_grid_df.sort_values("final_data_loss")
    axes[1].bar(loss_df["obs_scenario"], loss_df["final_data_loss"])
    axes[1].set_ylabel("Final normalized data loss")
    axes[1].set_title("Fit degradation under observation stress")
    axes[1].tick_params(axis="x", rotation=35)
    axes[1].grid(axis="y", alpha=0.25)

    fig.suptitle(f"Full-PINN observation stress, prior={CONFIG['selected_prior_scenario']}")
    fig.tight_layout()
    _img = FIGDIR / "full_pinn_observation_stress.png"
    fig.savefig(_img, dpi=180, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(_img)))

In [ ]:
# Heatmap panel for observation stress results.
if len(obs_grid_results):
    chosen = ["clean_full", "noisy_full", "weekly_sparse", "reporting_bias", "cases_only_proxy"]
    heatmaps = [("true C", C_true_np), ("prior C0", PRIOR_SCENARIOS[CONFIG["selected_prior_scenario"]]["C0"])]
    for obs_name in chosen:
        key = (obs_name, CONFIG["grid_seeds"][0])
        if key in obs_grid_results:
            heatmaps.append((obs_name, obs_grid_results[key]["C_hat"]))

    cols = 4
    rows = int(np.ceil(len(heatmaps) / cols))
    fig = plt.figure(figsize=(4.0 * cols + 0.8, 3.2 * rows))
    gs = fig.add_gridspec(rows, cols + 1, width_ratios=[1] * cols + [0.08], wspace=0.35, hspace=0.40)
    axes = [fig.add_subplot(gs[r, c]) for r in range(rows) for c in range(cols)]
    cax = fig.add_subplot(gs[:, -1])

    for ax, (title, M) in zip(axes, heatmaps):
        im = heatmap(ax, M, title)
    for ax in axes[len(heatmaps):]:
        ax.axis("off")
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label("coupling weight", rotation=270, labelpad=12)
    _img = FIGDIR / "full_pinn_observation_heatmaps.png"
    fig.savefig(_img, dpi=180, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(_img)))

## 7. Paper-ready interpretation

The full PINN is intentionally allowed to fit state trajectories flexibly. Therefore, a low trajectory or physics loss does **not** automatically mean that $C$ is identifiable. Use the grids above to report:

- $\|\hat C-C^*\|_F$ and max edge error,
- final data and physics losses,
- sensitivity to $C_0$ and $\lambda_C$,
- degradation under reporting bias,
- whether coupling estimates are prior-driven.

For the workshop paper, this supports a conservative claim: **geographic coupling inference from aggregate regional epidemic observations needs priors, reporting calibration, and sensitivity analysis.**

In [ ]:
# Export a compact summary table.
summary_parts = []
if len(prior_grid_df):
    s = prior_grid_df.copy()
    s["grid"] = "prior_sensitivity"
    summary_parts.append(s)
if len(obs_grid_df):
    s = obs_grid_df.copy()
    s["grid"] = "observation_stress"
    summary_parts.append(s)

if summary_parts:
    summary_df = pd.concat(summary_parts, ignore_index=True)
    summary_df.to_csv(OUTDIR / "full_pinn_summary_all_grids.csv", index=False)
    display_cols = [
        "grid", "obs_scenario", "prior_scenario", "alpha_hat", "alpha_abs_error",
        "C_fro_error", "C_max_abs_error", "final_data_loss", "final_phys_loss", "seconds",
    ]
    print("Saved:", OUTDIR / "full_pinn_summary_all_grids.csv")
    print("Figures:")
    for p in sorted(FIGDIR.glob("*.png")):
        print(" -", p)
    summary_df[display_cols].sort_values(["grid", "C_fro_error"])
else:
    print("No grids were run. Set run_prior_grid or run_observation_grid to True in CONFIG.")